# 06 — Generación de Embeddings

Generación de embeddings para cada criterio individual de inclusión y exclusión,
y para el texto de condiciones por estudio (primer filtro semántico en el pipeline online).

**Modelo criterios:** `pritamdeka/S-PubMedBert-MS-MARCO`  
PubMedBERT fine-tuneado sobre MS-MARCO: dominio clínico + optimizado para retrieval. 768 dims.

**Modelo condiciones:** `BAAI/bge-small-en-v1.5`  
Texto tipo keyword, filtro grueso, 384 dims.

**Granularidad:** 1 embedding por criterio individual  
Permite identificar exactamente qué criterio matchea con el perfil del paciente.

**Input:**
- `data/05_eligibility_normalized.parquet` -- criterios de inclusion/exclusion por estudio
- `data/05_clinical_trials_normalized.parquet` -- campo `condition_text` por estudio

**Output:**
- `data/06_inclusion_meta.parquet` / `06_inclusion_embeddings.npy`
- `data/06_exclusion_meta.parquet` / `06_exclusion_embeddings.npy`
- `data/06_condition_meta.parquet` / `06_condition_embeddings.npy`


## 1 — Setup


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm


c:\Users\orcor\anaconda3\envs\ds_cuda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_DIR = Path('../data')

# --- Input ---
ELIGIBILITY_PARQUET = DATA_DIR / '05_eligibility_normalized.parquet'
CLINICAL_PARQUET    = DATA_DIR / '05_clinical_trials_normalized.parquet'

# --- Models ---
CRITERIA_MODEL_NAME   = 'pritamdeka/S-PubMedBert-MS-MARCO'  # 768 dims
CONDITIONS_MODEL_NAME = 'BAAI/bge-small-en-v1.5'            # 384 dims

# --- Output ---
INCLUSION_META_PATH  = DATA_DIR / '06_inclusion_meta.parquet'
INCLUSION_EMB_PATH   = DATA_DIR / '06_inclusion_embeddings.npy'
EXCLUSION_META_PATH  = DATA_DIR / '06_exclusion_meta.parquet'
EXCLUSION_EMB_PATH   = DATA_DIR / '06_exclusion_embeddings.npy'
CONDITION_META_PATH  = DATA_DIR / '06_condition_meta.parquet'
CONDITION_EMB_PATH   = DATA_DIR / '06_condition_embeddings.npy'

# --- Processing ---
BATCH_SIZE       = 256
CHECKPOINT_EVERY = 50_000  # textos entre checkpoints


In [3]:
eligibility_df = pd.read_parquet(ELIGIBILITY_PARQUET)
clinical_df    = pd.read_parquet(CLINICAL_PARQUET)

print(f'Eligibility: {len(eligibility_df):,} estudios')
print(f'Clinical:    {len(clinical_df):,} estudios')
eligibility_df.head(2)


Eligibility: 83,962 estudios
Clinical:    83,962 estudios


,nct_id,enriched_inclusion,enriched_exclusion
0,NCT00728442,"[Non metastatic, including invasive and in sit...","[Breast disease without cancer, Metastatic bre..."
1,NCT01859442,"[diagnosis of T3N+ rectal cancer, above age of...",[unable to conduct a cardiopulmonary exercise ...


## 2 — Modelo


In [4]:
class EmbeddingModel:
    """Encapsula SentenceTransformer con soporte GPU y encoding normalizado."""

    def __init__(self, model_name: str) -> None:
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)
        self.model.to('cuda')
        print(f'Modelo cargado: {model_name}')
        print(f'Dimension embeddings: {self.model.get_sentence_embedding_dimension()}')

    def encode(
        self,
        texts: list[str],
        batch_size: int = 256,
        show_progress: bool = True,
    ) -> np.ndarray:
        """Codifica textos en batches, retorna array float32 normalizado."""
        return self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=show_progress,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )


## 3 — Embeddings de criterios (inclusion y exclusion)

1 embedding por criterio individual. Checkpointing cada `CHECKPOINT_EVERY` textos.


In [5]:
def explode_criteria(df: pd.DataFrame, col: str) -> pd.DataFrame:
    """Explota columna de listas a una fila por criterio individual."""
    exploded = (
        df[['nct_id', col]]
        .explode(col)
        .rename(columns={col: 'criterion_text'})
        .dropna(subset=['criterion_text'])
        .reset_index(drop=True)
    )
    exploded['criterion_index'] = exploded.groupby('nct_id').cumcount()
    return exploded


inclusion_meta = explode_criteria(eligibility_df, 'enriched_inclusion')
exclusion_meta = explode_criteria(eligibility_df, 'enriched_exclusion')

print(f'Criterios inclusion: {len(inclusion_meta):,}')
print(f'Criterios exclusion: {len(exclusion_meta):,}')


Criterios inclusion: 750,566
Criterios exclusion: 678,985


In [6]:
def embed_with_checkpoint(
    model: EmbeddingModel,
    texts: list[str],
    output_path: Path,
    batch_size: int = BATCH_SIZE,
    checkpoint_every: int = CHECKPOINT_EVERY,
) -> np.ndarray:
    """Embeds texts con checkpointing; retoma desde donde quedo si existe checkpoint."""
    checkpoint_path = output_path.with_suffix('.checkpoint.npy')

    start_idx = 0
    all_embeddings: list[np.ndarray] = []

    if checkpoint_path.exists():
        done = np.load(checkpoint_path)
        start_idx = len(done)
        all_embeddings.append(done)
        print(f'Retomando desde checkpoint: {start_idx:,} / {len(texts):,}')

    remaining = texts[start_idx:]

    for i in tqdm(range(0, len(remaining), checkpoint_every), desc='Chunks'):
        chunk = remaining[i : i + checkpoint_every]
        emb = model.encode(chunk, batch_size=batch_size, show_progress=True)
        all_embeddings.append(emb)
        np.save(checkpoint_path, np.vstack(all_embeddings))

    final = np.vstack(all_embeddings) if all_embeddings else np.empty((0,))
    np.save(output_path, final)
    checkpoint_path.unlink(missing_ok=True)
    print(f'Guardado: {output_path} -- shape {final.shape}')
    return final


In [7]:
criteria_model = EmbeddingModel(CRITERIA_MODEL_NAME)

inclusion_embeddings = embed_with_checkpoint(
    criteria_model,
    inclusion_meta['criterion_text'].tolist(),
    INCLUSION_EMB_PATH,
)
inclusion_meta.to_parquet(INCLUSION_META_PATH, index=False)
print(f'Meta guardada: {INCLUSION_META_PATH}')

exclusion_embeddings = embed_with_checkpoint(
    criteria_model,
    exclusion_meta['criterion_text'].tolist(),
    EXCLUSION_EMB_PATH,
)
exclusion_meta.to_parquet(EXCLUSION_META_PATH, index=False)
print(f'Meta guardada: {EXCLUSION_META_PATH}')


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 60316.99it/s]
C:\Users\orcor\AppData\Local\Temp\ipykernel_12104\2612555553.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f'Dimension embeddings: {self.model.get_sentence_embedding_dimension()}')


Modelo cargado: pritamdeka/S-PubMedBert-MS-MARCO
Dimension embeddings: 768


Chunks: 100%|██████████| 16/16 [10:51<00:00, 40.72s/it]


Guardado: ..\data\06_inclusion_embeddings.npy -- shape (750566, 768)
Meta guardada: ..\data\06_inclusion_meta.parquet


Chunks: 100%|██████████| 14/14 [08:52<00:00, 38.04s/it]


Guardado: ..\data\06_exclusion_embeddings.npy -- shape (678985, 768)
Meta guardada: ..\data\06_exclusion_meta.parquet


## 4 — Embeddings de condiciones (por estudio)

1 embedding por estudio sobre `condition_text` (MeSH terms + condiciones raw).  
Primer filtro semantico grueso en el pipeline online. ~84k textos, sin checkpointing.


In [8]:
conditions_model = EmbeddingModel(CONDITIONS_MODEL_NAME)

condition_meta = clinical_df[['nct_id', 'condition_text']].copy()

condition_embeddings = conditions_model.encode(
    condition_meta['condition_text'].tolist(),
    batch_size=BATCH_SIZE,
)

np.save(CONDITION_EMB_PATH, condition_embeddings)
condition_meta.to_parquet(CONDITION_META_PATH, index=False)

print(f'Condition embeddings: {CONDITION_EMB_PATH} -- shape {condition_embeddings.shape}')
print(f'Condition meta:       {CONDITION_META_PATH}')


c:\Users\orcor\anaconda3\envs\ds_cuda\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\orcor\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15522.33it/s]
C:\Users\orcor\AppData\Local\Temp\ipyk

Modelo cargado: BAAI/bge-small-en-v1.5
Dimension embeddings: 384


Batches: 100%|██████████| 328/328 [00:24<00:00, 13.20it/s]


Condition embeddings: ..\data\06_condition_embeddings.npy -- shape (83962, 384)
Condition meta:       ..\data\06_condition_meta.parquet


## 5 — Verificacion

Comprueba alineacion metadata<->embeddings y sanity check de similitudes.


In [9]:
inc_emb  = np.load(INCLUSION_EMB_PATH)
exc_emb  = np.load(EXCLUSION_EMB_PATH)
con_emb  = np.load(CONDITION_EMB_PATH)
inc_meta = pd.read_parquet(INCLUSION_META_PATH)
exc_meta = pd.read_parquet(EXCLUSION_META_PATH)
con_meta = pd.read_parquet(CONDITION_META_PATH)

assert len(inc_emb) == len(inc_meta), 'Mismatch inclusion'
assert len(exc_emb) == len(exc_meta), 'Mismatch exclusion'
assert len(con_emb) == len(con_meta), 'Mismatch condiciones'

print('=== Shapes ===')
print(f'Inclusion:   {inc_emb.shape}  OK')
print(f'Exclusion:   {exc_emb.shape}  OK')
print(f'Condiciones: {con_emb.shape}  OK')


def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    """Similitud coseno entre vectores ya normalizados (dot = cosine)."""
    return float(np.dot(a, b))


print('\n=== Similitudes de muestra (criterios de inclusion) ===')
samples = inc_meta['criterion_text'].iloc[:4].tolist()
for i, t in enumerate(samples[:3]):
    print(f'\n[{i}] {t[:90]}')
    for j in range(i + 1, 4):
        sim = cosine_sim(inc_emb[i], inc_emb[j])
        snippet = samples[j][:60]
        print(f'    vs [{j}] "{snippet}" -> {sim:.4f}')


=== Shapes ===
Inclusion:   (750566, 768)  OK
Exclusion:   (678985, 768)  OK
Condiciones: (83962, 384)  OK

=== Similitudes de muestra (criterios de inclusion) ===

[0] Non metastatic, including invasive and in situ, breast cancers as well as axillary cancer 
    vs [1] "At least one therapeutic MSM decision." -> 0.8323
    vs [2] "diagnosis of T3N+ rectal cancer" -> 0.8766
    vs [3] "above age of 18 [demographics]" -> 0.8540

[1] At least one therapeutic MSM decision.
    vs [2] "diagnosis of T3N+ rectal cancer" -> 0.8205
    vs [3] "above age of 18 [demographics]" -> 0.8385

[2] diagnosis of T3N+ rectal cancer
    vs [3] "above age of 18 [demographics]" -> 0.8265


## 6 — Modelos alternativos (opcional)

Descomentar para generar con modelos alternativos.  
Cada uno guarda en archivos separados con sufijo del nombre del modelo.


In [17]:
# =============================================================================
# ALT criterios: neuml/pubmedbert-base-embeddings
# Mismo dominio clinico, entrenado como sentence embedder general (no retrieval).
# =============================================================================

ALT_CRITERIA_MODEL = 'neuml/pubmedbert-base-embeddings'
alt_criteria_model = EmbeddingModel(ALT_CRITERIA_MODEL)
suffix = ALT_CRITERIA_MODEL.replace('/', '__')

embed_with_checkpoint(
    alt_criteria_model,
    inclusion_meta['criterion_text'].tolist(),
    DATA_DIR / f'06_inclusion_embeddings_{suffix}.npy',
)
embed_with_checkpoint(
    alt_criteria_model,
    exclusion_meta['criterion_text'].tolist(),
    DATA_DIR / f'06_exclusion_embeddings_{suffix}.npy',
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9721.14it/s]
C:\Users\orcor\AppData\Local\Temp\ipykernel_12104\2612555553.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f'Dimension embeddings: {self.model.get_sentence_embedding_dimension()}')


Modelo cargado: neuml/pubmedbert-base-embeddings
Dimension embeddings: 768


Chunks: 100%|██████████| 16/16 [11:01<00:00, 41.33s/it]


Guardado: ..\data\06_inclusion_embeddings_neuml__pubmedbert-base-embeddings.npy -- shape (750566, 768)


Chunks: 100%|██████████| 14/14 [09:02<00:00, 38.77s/it]


Guardado: ..\data\06_exclusion_embeddings_neuml__pubmedbert-base-embeddings.npy -- shape (678985, 768)


array([[ 0.00030336,  0.00788502,  0.06837899, ..., -0.01720062,
         0.05659734, -0.01367811],
       [-0.00998278, -0.00630648, -0.01885159, ...,  0.04973661,
        -0.07230275, -0.01048777],
       [ 0.0240372 , -0.01774266,  0.02064039, ...,  0.03727059,
         0.09041013,  0.00986875],
       ...,
       [-0.01796798, -0.03577607,  0.01781044, ...,  0.07616588,
         0.03891111, -0.01752861],
       [-0.01051624,  0.04968359, -0.02025015, ...,  0.02938726,
        -0.05665486, -0.00368907],
       [-0.00417612,  0.0320162 , -0.00682746, ..., -0.00273863,
        -0.04588433, -0.01025742]], shape=(678985, 768), dtype=float32)

In [ ]:
# =============================================================================
# ALT criterios: all-MiniLM-L6-v2
# General purpose, 384 dims, muy rapido. Baseline de referencia.
# =============================================================================

ALT_CRITERIA_MODEL = 'all-MiniLM-L6-v2'
alt_criteria_model = EmbeddingModel(ALT_CRITERIA_MODEL)
suffix = ALT_CRITERIA_MODEL.replace('/', '__')

embed_with_checkpoint(
    alt_criteria_model,
    inclusion_meta['criterion_text'].tolist(),
    DATA_DIR / f'06_inclusion_embeddings_{suffix}.npy',
)
embed_with_checkpoint(
    alt_criteria_model,
    exclusion_meta['criterion_text'].tolist(),
    DATA_DIR / f'06_exclusion_embeddings_{suffix}.npy',
)


c:\Users\orcor\anaconda3\envs\ds_cuda\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\orcor\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8703.10it/s]
C:\Users\orcor\AppData\

Modelo cargado: all-MiniLM-L6-v2
Dimension embeddings: 384


Chunks: 100%|██████████| 16/16 [03:13<00:00, 12.09s/it]


Guardado: ..\data\06_inclusion_embeddings_all-MiniLM-L6-v2.npy -- shape (750566, 384)


Chunks: 100%|██████████| 14/14 [02:36<00:00, 11.20s/it]


Guardado: ..\data\06_exclusion_embeddings_all-MiniLM-L6-v2.npy -- shape (678985, 384)


array([[ 0.0058782 ,  0.02577103, -0.08366734, ..., -0.05825529,
         0.10667596,  0.04815762],
       [ 0.04369266, -0.00454328, -0.04248426, ..., -0.04185404,
         0.11799555,  0.02979443],
       [ 0.13446052, -0.01011006, -0.06418625, ..., -0.10591672,
         0.06367377,  0.01630351],
       ...,
       [ 0.00016962,  0.03106358,  0.00885624, ..., -0.11098701,
         0.0499125 ,  0.0301625 ],
       [ 0.01100497,  0.08890112, -0.05464414, ...,  0.09412808,
         0.01327624, -0.06132923],
       [-0.01060996, -0.01815983, -0.05898078, ..., -0.00603302,
         0.07402932,  0.04057718]], shape=(678985, 384), dtype=float32)

In [19]:
# =============================================================================
# ALT condiciones: S-PubMedBert-MS-MARCO y all-MiniLM-L6-v2
# =============================================================================

for alt_cond_model_name in [
    'pritamdeka/S-PubMedBert-MS-MARCO',
    'all-MiniLM-L6-v2',
]:
    alt_cond_model = EmbeddingModel(alt_cond_model_name)
    suffix = alt_cond_model_name.replace('/', '__')
    alt_con_emb = alt_cond_model.encode(
        condition_meta['condition_text'].tolist(), batch_size=BATCH_SIZE
    )
    out_path = DATA_DIR / f'06_condition_embeddings_{suffix}.npy'
    np.save(out_path, alt_con_emb)
    print(f'Guardado {alt_cond_model_name}: shape {alt_con_emb.shape}')


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 50674.91it/s]
C:\Users\orcor\AppData\Local\Temp\ipykernel_12104\2612555553.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f'Dimension embeddings: {self.model.get_sentence_embedding_dimension()}')


Modelo cargado: pritamdeka/S-PubMedBert-MS-MARCO
Dimension embeddings: 768


Batches: 100%|██████████| 328/328 [00:52<00:00,  6.23it/s]


Guardado pritamdeka/S-PubMedBert-MS-MARCO: shape (83962, 768)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7067.36it/s]
C:\Users\orcor\AppData\Local\Temp\ipykernel_12104\2612555553.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f'Dimension embeddings: {self.model.get_sentence_embedding_dimension()}')


Modelo cargado: all-MiniLM-L6-v2
Dimension embeddings: 384


Batches: 100%|██████████| 328/328 [00:13<00:00, 23.74it/s]


Guardado all-MiniLM-L6-v2: shape (83962, 384)
